In [1]:
import os
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import fisher_exact, chi2_contingency, MonteCarloMethod
from scipy.stats.contingency import expected_freq
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Table, TableStyle, Spacer, PageBreak

OUT = 'D2_daily_watch_time'
ORDER = ['Less than 1 hour', '1 hour', '2 hours', '3 hours', 'More than that']
BASE = os.getcwd()  # project root (notebook's folder)
PDF_PATH = os.path.join(BASE, 'fisher_test', 'fisher_test_D2_daily_watch_time.pdf')
N_RESAMPLES = 100_000  # SciPy computes r x c Fisher p-values by Monte Carlo
SEED = 42              # fixed seed so p-values are reproducible

d = pd.read_pickle(os.path.join(BASE, 'dataset', 'dataset_v3.pkl'))
d[OUT] = pd.Categorical(d[OUT], categories=ORDER, ordered=True)
preds = [c for c in d.columns if c != OUT]

# Fisher-Freeman-Halton when any cell has an expected count of 5 or less;
# chi-square otherwise
results = {}
for c in preds:
    ct = pd.crosstab(d[c], d[OUT])
    e = expected_freq(ct)
    low = (e <= 5).mean() * 100
    if (e <= 5).any():
        test = 'Fisher-Freeman-Halton'
        p = fisher_exact(ct.values, method=MonteCarloMethod(n_resamples=N_RESAMPLES, rng=np.random.default_rng(SEED))).pvalue
    else:
        test = 'Chi-square'
        p = chi2_contingency(ct)[1]
    results[c] = dict(ct=ct, pct=ct.div(ct.sum(1), axis=0) * 100, low=low, test=test, p=p)

summary_df = pd.DataFrame(
    [(c, r['ct'].shape[0], r['low'], r['test'], r['p'], r['p'] < 0.05) for c, r in results.items()],
    columns=['Variable', 'Categories', '% expected <= 5', 'Test', 'p-value', 'Significant'])
summary_df

,Variable,Categories,% expected <= 5,Test,p-value,Significant
0,A1_age_group,3,6.666667,Fisher-Freeman-Halton,2.878871e-01,False
1,A2_gender,2,0.000000,Chi-square,1.130213e-01,False
2,A3_religion,2,50.000000,Fisher-Freeman-Halton,4.331957e-01,False
3,A4_marital_status,3,33.333333,Fisher-Freeman-Halton,7.209928e-03,True
4,A5_residence,2,0.000000,Chi-square,4.474249e-01,False
5,A6_occupation,4,0.000000,Chi-square,2.314373e-01,False
6,B1_family_type,4,50.000000,Fisher-Freeman-Halton,3.126969e-01,False
7,B2_family_time_group,5,0.000000,Chi-square,1.813063e-02,True
8,B3_family_relationship,4,25.000000,Fisher-Freeman-Halton,1.236988e-02,True
9,B4_in_person_feeling,5,24.000000,Fisher-Freeman-Halton,5.069949e-03,True


In [2]:
# Variables tested with Fisher-Freeman-Halton
summary_df[summary_df['Test'] == 'Fisher-Freeman-Halton']

,Variable,Categories,% expected <= 5,Test,p-value,Significant
0,A1_age_group,3,6.666667,Fisher-Freeman-Halton,0.287887,False
2,A3_religion,2,50.000000,Fisher-Freeman-Halton,0.433196,False
3,A4_marital_status,3,33.333333,Fisher-Freeman-Halton,0.007210,True
6,B1_family_type,4,50.000000,Fisher-Freeman-Halton,0.312697,False
8,B3_family_relationship,4,25.000000,Fisher-Freeman-Halton,0.012370,True
9,B4_in_person_feeling,5,24.000000,Fisher-Freeman-Halton,0.005070,True
11,C2_weight_group,5,4.000000,Fisher-Freeman-Halton,0.033060,True
15,D3_peak_watch_time,6,46.666667,Fisher-Freeman-Halton,0.025180,True
20,E4_substance_use,5,48.000000,Fisher-Freeman-Halton,0.112359,False
21,F1_attention_decreased,4,5.000000,Fisher-Freeman-Halton,0.000010,True


In [3]:
ss = getSampleStyleSheet()
fmt_p = lambda p: '< 0.0001' if p < 0.0001 else f'{p:.4f}'

def tstyle(header_color):
    return TableStyle([('BACKGROUND', (0, 0), (-1, 0), header_color), ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTSIZE', (0, 0), (-1, -1), 8), ('GRID', (0, 0), (-1, -1), 0.4, colors.grey),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'), ('ALIGN', (1, 0), (-1, -1), 'CENTER'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f2f5f9')])])

el = [Paragraph('Association Tests: Chi-square and Fisher-Freeman-Halton', ss['Title']),
      Paragraph(f'Outcome variable: <b>{OUT}</b> (columns) vs. each variable (rows). N = {len(d)}. '
                'The Fisher-Freeman-Halton test was used when any cell had an expected count of 5 or less; '
                'otherwise the chi-square test of independence was used. Fisher-Freeman-Halton p-values were '
                f'computed with SciPy (Monte Carlo, {N_RESAMPLES:,} samples, seed {SEED}).', ss['Normal']),
      Spacer(1, 10), Paragraph('Summary of association tests (* p &lt; 0.05)', ss['Heading2'])]

srows = [['Variable', '% expected <= 5', 'Test', 'p-value', 'Sig.']] + [
    [c, f"{r['low']:.1f}%", r['test'], fmt_p(r['p']), '*' if r['p'] < 0.05 else ''] for c, r in results.items()]
t = Table(srows, repeatRows=1, colWidths=[220, 90, 160, 90, 40]); t.setStyle(tstyle(colors.HexColor('#2c3e50')))
el += [t, PageBreak()]

for c, r in results.items():
    ct, pct = r['ct'], r['pct']
    rows = [[c] + ORDER + ['Total']]
    for idx in ct.index:
        rows.append([str(idx)] + [f'{ct.loc[idx, o]} ({pct.loc[idx, o]:.1f}%)' for o in ORDER] + [str(ct.loc[idx].sum())])
    tot = ct.sum()
    rows.append(['Total'] + [f'{tot[o]} ({tot[o]/len(d)*100:.1f}%)' for o in ORDER] + [str(len(d))])
    tb = Table(rows, colWidths=[170] + [100] * 5 + [50], repeatRows=1); tb.setStyle(tstyle(colors.HexColor('#1f6f8b')))
    el += [Paragraph(f'{c} × {OUT}', ss['Heading3']), tb,
           Paragraph(f"{r['test']}: p = {fmt_p(r['p'])}" + (' (significant)' if r['p'] < 0.05 else '')
                     + f" | Cells with expected count &lt;= 5: {r['low']:.1f}%", ss['Normal']),
           Spacer(1, 14)]

os.makedirs(os.path.dirname(PDF_PATH), exist_ok=True)
SimpleDocTemplate(PDF_PATH, pagesize=landscape(A4), leftMargin=30, rightMargin=30, topMargin=30, bottomMargin=30).build(el)
print('Saved', PDF_PATH, '-', len(results), 'tests')

Saved /Users/ziban/github/Reels-Mental-Health-Well-Being-Report/fisher_test/fisher_test_D2_daily_watch_time.pdf - 40 tests
